# Review-Based Recommendation Explanation (Sephora)
**Course:** Natural Language and Responsive AI (TEE7022) | **Assessment:** Continuous Assessment-IV

**Goal:** a customer says what they want ("hydrating, lightweight and good value"). The system recommends a product and **explains why using real sentences from customer reviews**.

**Data:** Sephora product catalogue (`product_info.csv`) + customer reviews (`reviews_0-250.csv` from the Kaggle dataset *Sephora Products and Skincare Reviews*).

The notebook is built **phase by phase**. Each phase is one NLP idea from the syllabus, with a plain-English explanation, then code, then output.

| Phase | What happens | Syllabus unit |
|---|---|---|
| 0 | Load and join the data | - |
| 1 | Clean the raw text (regular expressions) | 1, 2 |
| 2 | Sentence segmentation and tokenisation | 1, 2 |
| 3 | Stop-word removal (keeping negations) | 1, 2 |
| 4 | Stemming vs lemmatisation (morphology) | 2 |
| 5 | Part-of-speech (POS) tagging | 2 |
| 6 | Unigrams, bigrams, trigrams, smoothing | 2 |
| 7 | Bag of words | 3 |
| 8 | Lexical semantics: the aspect lexicon | 3 |
| 9 | Sentiment per sentence | 3 |
| 10 | Ranking products for a request | - |
| 11 | Evidence-backed explanation | - |
| 12 | Evaluation | - |
| 13 | Live demo | - |

## Setup

In [31]:
!pip install -q vaderSentiment simplemma textblob scikit-learn pandas nltk scipy

In [32]:
import re, glob, os, html
from collections import Counter
from functools import lru_cache

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import simplemma                                              # lemmatiser (no downloads needed)
from textblob.en import tag as pos_tag                        # POS tagger (no downloads needed)
from nltk.stem import SnowballStemmer                         # stemmer (no downloads needed)
from nltk.util import ngrams                                  # n-gram helper
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 130)
analyzer = SentimentIntensityAnalyzer()
stemmer = SnowballStemmer("english")

## Phase 0: Load and join the data

**Plain English:** we have two tables. One lists products (name, brand, price, category). The other lists reviews (text, stars, which product). We join them on `product_id`.

To keep the notebook fast, we keep the **150 products with the most reviews** and at most **150 reviews per product**. Change the numbers below if you want more.


In [33]:
import csv

PRODUCT_FILE   = "product_info.csv"
REVIEW_PATTERN = "reviews_0-250.csv"      # matches reviews_0-250.csv, reviews_0_250.csv, ...
TOP_PRODUCTS   = 150                  # keep this many products (most reviewed)
MAX_PER_PRODUCT = 150                 # keep at most this many reviews per product
MIN_CHARS      = 25                   # drop very short reviews
SEED           = 42

if not os.path.exists(PRODUCT_FILE) or not glob.glob(REVIEW_PATTERN):
    try:
        from google.colab import files
        print("Please upload product_info.csv and at least one reviews_*.csv")
        files.upload()
    except ImportError:
        raise FileNotFoundError("Put product_info.csv and reviews_*.csv in this folder.")

products = pd.read_csv(PRODUCT_FILE, usecols=[
    "product_id", "product_name", "brand_name", "rating", "reviews", "price_usd",
    "primary_category", "secondary_category", "tertiary_category"
]).rename(columns={"rating": "avg_rating", "reviews": "n_reviews_total"})

WANT = {"product_id", "rating", "review_text", "review_title", "is_recommended", "skin_type"}
frames = [pd.read_csv(f, usecols=lambda c: c in WANT, engine='python', on_bad_lines='skip')
          for f in sorted(glob.glob(REVIEW_PATTERN))]
reviews = pd.concat(frames, ignore_index=True).rename(columns={"rating": "stars"})
print("files read:", len(frames), "| raw reviews:", len(reviews))

files read: 1 | raw reviews: 292774


In [34]:
# keep usable reviews only
reviews = reviews.dropna(subset=["review_text", "stars"])
reviews["review_text"] = reviews["review_text"].astype(str)
reviews = reviews[reviews["review_text"].str.len() >= MIN_CHARS]
reviews = reviews.drop_duplicates(subset=["product_id", "review_text"])
reviews = reviews[reviews["product_id"].isin(products["product_id"])]          # join check

# keep the most-reviewed products, then cap reviews per product (shuffled, so it is a random sample)
top_ids = reviews["product_id"].value_counts().head(TOP_PRODUCTS).index
reviews = reviews[reviews["product_id"].isin(top_ids)]
reviews = reviews.sample(frac=1, random_state=SEED).groupby("product_id").head(MAX_PER_PRODUCT)
reviews = reviews.reset_index(drop=True)
reviews["review_id"] = reviews.index

data = reviews.merge(products, on="product_id", how="left")
print("reviews kept:", len(data), "| products kept:", data["product_id"].nunique())
print()
print("Categories available (use these in Phase 13):")
print(data.drop_duplicates("product_id")["tertiary_category"].value_counts().head(8).to_string())
data[["review_id", "product_name", "stars", "review_text"]].head(5)

reviews kept: 10201 | products kept: 69

Categories available (use these in Phase 13):
tertiary_category
Moisturizers             15
Face Wash & Cleansers    10
Face Serums               7
Toners                    3
Face Masks                3
Mists & Essences          3
Facial Peels              2
Face Sunscreen            2


,review_id,product_name,stars,review_text
0,0,Green Clean Makeup Meltaway Cleansing Balm Limited Edition Jumbo,5,I’m obsessed with this stuff! I feel like a lot of cleaning balms ...
1,1,Mini The Water Cream,5,"I have combo skin, dryer in the winter, oiler in the summer. Tatch..."
2,2,Rosebud Salve,5,"I absolutely love this product! Perfect for lips, cuticles and dry..."
3,3,Unseen Sunscreen SPF 40 PA+++,5,"This is, without a doubt, the best sunscreen I’ve ever used. Leave..."
4,4,Mini Cicapair Tiger Grass Color Correcting Treatment SPF 30,5,This is an interesting product. I wouldn’t say my skin is overly r...


**Optional (if uploading a big file is slow):** download straight from Kaggle inside Colab. Check the dataset name on Kaggle first. You need to be logged in to Kaggle in Colab for this to work. Then point `PRODUCT_FILE` and `REVIEW_PATTERN` at the downloaded folder.

In [35]:
# import kagglehub
# path = kagglehub.dataset_download("nadyinky/sephora-products-and-skincare-reviews")   # check the exact name on Kaggle
# print(path); print(os.listdir(path))
# PRODUCT_FILE   = os.path.join(path, "product_info.csv")
# REVIEW_PATTERN = os.path.join(path, "reviews_0-250.csv")

## Phase 1: Clean the raw text (regular expressions)

**Plain English:** real reviews are messy: HTML tags, web links, `soooooo`, `!!!!!!`. We clean these with **regular expressions** (patterns that find text). We keep upper/lower case and punctuation here, because the sentiment tool in Phase 9 uses them (for example, `LOVE!!` is stronger than `love`).

| Pattern | Meaning |
|---|---|
| `<[^>]+>` | anything inside `< >` (HTML tags) |
| `https?://\S+` | a web link |
| `(.)\1{3,}` | the same character 4+ times in a row (`soooooo`) |
| `([!?.]){2,}` | repeated punctuation (`!!!!`) |

In [36]:
def clean_text(text):
    t = html.unescape(text)                       # &amp; -> &
    t = re.sub(r"<[^>]+>", " ", t)                # remove HTML tags
    t = re.sub(r"https?://\S+", " ", t)           # remove links
    t = t.replace("\u2019", "'")                  # curly apostrophe -> normal
    t = re.sub(r"(.)\1{3,}", r"\1\1", t)          # soooooo -> soo
    t = re.sub(r"([!?.])\1+", r"\1", t)           # !!!! -> !
    t = re.sub(r"\s+", " ", t).strip()            # tidy spaces
    return t

demo = ["Loooove this!!!!! <br /> soooo hydrating... check https://sephora.com &amp; buy it",
        "It DOESN'T work for me???  Waste of money"]
for d in demo:
    print("BEFORE:", d)
    print("AFTER :", clean_text(d))
    print()

BEFORE: Loooove this!!!!! <br /> soooo hydrating... check https://sephora.com &amp; buy it
AFTER : Loove this! soo hydrating. check & buy it

BEFORE: It DOESN'T work for me???  Waste of money
AFTER : It DOESN'T work for me? Waste of money



## Phase 2: Sentence segmentation and tokenisation

**Plain English:**
- **Sentence segmentation** cuts a review into sentences. We do this because one review can praise one thing and criticise another ("Great scent but it broke me out"). Judging sentence by sentence lets us quote the exact sentence as evidence.
- **Tokenisation** cuts a sentence into words (tokens). First we expand contractions, so `doesn't` becomes `does not`. This matters because **negation flips meaning**.

The sentence cut is a regular expression: split after `.` `!` `?` followed by a space.

In [37]:
def split_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+", text)
    return [p.strip() for p in parts if len(p.strip()) >= 3]

def expand_contractions(s):
    s = s.lower().replace("\u2019", "'")
    s = re.sub(r"won't", "will not", s)
    s = re.sub(r"can't|cannot", "can not", s)
    s = re.sub(r"n't\b", " not", s)
    s = re.sub(r"'re\b", " are", s)
    s = re.sub(r"'ve\b", " have", s)
    s = re.sub(r"'ll\b", " will", s)
    s = re.sub(r"'m\b", " am", s)
    s = re.sub(r"'d\b", " would", s)
    s = re.sub(r"'s\b", "", s)
    return s

def tokenize(sentence):
    return re.findall(r"[a-z]{2,}", expand_contractions(sentence))

example = "It doesn't moisturize enough. But the scent is lovely! Won't repurchase."
print(split_sentences(example))
print(tokenize("It doesn't moisturize enough."))

["It doesn't moisturize enough.", 'But the scent is lovely!', "Won't repurchase."]
['it', 'does', 'not', 'moisturize', 'enough']


In [38]:
# build the sentence table: one row per sentence, keeping which review and product it came from
rows = []
for rid, pid, stars, text in zip(data["review_id"], data["product_id"], data["stars"], data["review_text"]):
    for s in split_sentences(clean_text(text)):
        rows.append((rid, pid, stars, s))
sents = pd.DataFrame(rows, columns=["review_id", "product_id", "stars", "sentence"])
sents["tokens"] = sents["sentence"].apply(tokenize)
sents = sents[sents["tokens"].str.len() > 0].reset_index(drop=True)
print(len(data), "reviews ->", len(sents), "sentences")
sents[["product_id", "stars", "sentence", "tokens"]].head(5)

10201 reviews -> 43590 sentences


,product_id,stars,sentence,tokens
0,P450271,5,I'm obsessed with this stuff!,"[am, obsessed, with, this, stuff]"
1,P450271,5,I feel like a lot of cleaning balms are a bit tedious to get off d...,"[feel, like, lot, of, cleaning, balms, are, bit, tedious, to, get,..."
2,P450271,5,"It so so easy to use, I love it!","[it, so, so, easy, to, use, love, it]"
3,P454794,5,"I have combo skin, dryer in the winter, oiler in the summer.","[have, combo, skin, dryer, in, the, winter, oiler, in, the, summer]"
4,P454794,5,Tatcha is one of the very few brands with fragrance that my skin c...,"[tatcha, is, one, of, the, very, few, brands, with, fragrance, tha..."


## Phase 3: Stop-word removal

**Plain English:** words like *the, is, and, of* are very common and carry little meaning for our job. Removing them leaves the **content words** (*hydrating, scent, price*).

**Careful:** the standard stop-word list also contains **negations** (*not, no, never, without*). If we delete them, "not moisturizing" becomes "moisturizing" and the meaning flips. So we remove the negations (and *very, too, enough*) **from** the stop-word list, which keeps them.

In [39]:
KEEP = {"not", "no", "never", "nor", "none", "nothing", "nobody", "cannot", "without", "very", "too", "enough", "hardly", "barely"}
STOP = set(ENGLISH_STOP_WORDS) - KEEP

def remove_stop(tokens):
    return [t for t in tokens if t not in STOP]

before = tokenize("The scent is not very strong and it does not irritate my skin at all.")
print("before:", before)
print("after :", remove_stop(before))

sents["content"] = sents["tokens"].apply(remove_stop)
all_before = sum(len(t) for t in sents["tokens"])
all_after = sum(len(t) for t in sents["content"])
print(f"\ntokens: {all_before:,} -> {all_after:,}  ({100 * (1 - all_after / all_before):.0f}% removed)")

before: ['the', 'scent', 'is', 'not', 'very', 'strong', 'and', 'it', 'does', 'not', 'irritate', 'my', 'skin', 'at', 'all']
after : ['scent', 'not', 'very', 'strong', 'does', 'not', 'irritate', 'skin']

tokens: 551,995 -> 276,599  (50% removed)


## Phase 4: Stemming vs lemmatisation (morphology)

**Plain English:** the same idea appears in many word forms: *moisturize, moisturizing, moisturizes*. We need to treat them as one.

| | Stemming | Lemmatisation |
|---|---|---|
| How | Chops off endings by rules | Looks the word up in a dictionary |
| Result | May not be a real word (`moistur`) | Always a real word (`moisturize`) |
| Speed | Faster | Slower |

We use **lemmatisation** in this project because the lemmas are real words that we can put in a readable lexicon. The table below compares both.

*Known limitation:* the lemmatiser ignores context, so it sometimes picks the wrong dictionary form (for example, *worse* becomes *wrong*).

In [40]:
@lru_cache(maxsize=None)
def lemma(word):
    return simplemma.lemmatize(word, lang="en")

demo_words = ["moisturizing", "moisturizes", "hydrated", "hydrating", "lasted", "lasting",
              "better", "skins", "breakouts", "irritated", "leaves", "drier", "smells", "creamier", "absorbs"]
pd.DataFrame({"word": demo_words,
              "stem (rule-based)": [stemmer.stem(w) for w in demo_words],
              "lemma (dictionary)": [lemma(w) for w in demo_words]})

,word,stem (rule-based),lemma (dictionary)
0,moisturizing,moistur,moisturize
1,moisturizes,moistur,moisturize
2,hydrated,hydrat,hydrate
3,hydrating,hydrat,hydrate
4,lasted,last,last
5,lasting,last,last
6,better,better,good
7,skins,skin,skin
8,breakouts,breakout,breakout
9,irritated,irrit,irritate


In [41]:
sents["lemmas"] = sents["content"].apply(lambda toks: [lemma(t) for t in toks])
sents[["sentence", "content", "lemmas"]].head(5)

,sentence,content,lemmas
0,I'm obsessed with this stuff!,"[obsessed, stuff]","[obsess, stuff]"
1,I feel like a lot of cleaning balms are a bit tedious to get off d...,"[feel, like, lot, cleaning, balms, bit, tedious, oily, makeup, rem...","[feel, like, lot, clean, balm, bit, tedious, oily, makeup, remover..."
2,"It so so easy to use, I love it!","[easy, use, love]","[easy, use, love]"
3,"I have combo skin, dryer in the winter, oiler in the summer.","[combo, skin, dryer, winter, oiler, summer]","[combo, skin, dryer, winter, oiler, summer]"
4,Tatcha is one of the very few brands with fragrance that my skin c...,"[tatcha, very, brands, fragrance, skin, handle]","[tatcha, very, brand, fragrance, skin, handle]"


## Phase 5: Part-of-speech (POS) tagging

**Plain English:** POS tagging labels each word as a noun, verb, adjective and so on. Tags used here (Penn Treebank):

| Tag | Meaning | Example |
|---|---|---|
| NN / NNS | noun | skin, scent |
| VB / VBZ | verb | feel, smells |
| JJ / JJR / JJS | adjective | soft, lightweight, best |
| RB | adverb | very, never |

**How we use it:** adjectives are the *opinion words*. In Phase 11 we pick out the adjectives from the evidence sentences, so the explanation can say *"reviewers describe it as creamy, lightweight"*.

In [42]:
for s in ["This cream makes my dry skin feel very soft and smells lovely.",
          "It is not moisturizing enough for my dry skin."]:
    print(pos_tag(s))
    print()

def opinion_adjectives(sentence):
    return [lemma(w.lower()) for w, t in pos_tag(sentence) if t in ("JJ", "JJR", "JJS") and w.lower() not in STOP]

print(opinion_adjectives("The texture is silky, lightweight and never greasy."))

[('This', 'DT'), ('cream', 'NN'), ('makes', 'VBZ'), ('my', 'PRP$'), ('dry', 'JJ'), ('skin', 'NN'), ('feel', 'VB'), ('very', 'RB'), ('soft', 'JJ'), ('and', 'CC'), ('smells', 'VBZ'), ('lovely', 'JJ'), ('.', '.')]

[('It', 'PRP'), ('is', 'VBZ'), ('not', 'RB'), ('moisturizing', 'VBG'), ('enough', 'RB'), ('for', 'IN'), ('my', 'PRP$'), ('dry', 'JJ'), ('skin', 'NN'), ('.', '.')]

['silky', 'lightweight', 'greasy']


## Phase 6: Unigrams, bigrams, trigrams and smoothing

**Plain English:** an **n-gram** is n words in a row. *dry* is a unigram, *dry skin* a bigram, *fine lines appear* a trigram. Counting them shows which phrases people really use, and it is the basis of simple **language models**.

**Smoothing:** a bigram model estimates `P(next word | word)` = count(word, next) / count(word). A pair never seen in the data would get probability **0**, which is too harsh. **Add-one (Laplace) smoothing** adds 1 to every count so unseen pairs get a small non-zero probability:

`P(w2 | w1) = (count(w1, w2) + 1) / (count(w1) + V)`  where V is the vocabulary size.

In [43]:
sample = sents["tokens"].sample(min(30000, len(sents)), random_state=SEED)

uni = Counter(w for toks in sample for w in toks)
bi  = Counter(b for toks in sample for b in ngrams(toks, 2))
tri = Counter(t for toks in sample for t in ngrams(toks, 3))

def has_content(gram):                 # skip grams made only of stop words
    return any(w not in STOP for w in gram)

print("Top bigrams:")
print([(" ".join(g), c) for g, c in bi.most_common(60) if has_content(g)][:15])
print("\nTop trigrams:")
print([(" ".join(g), c) for g, c in tri.most_common(60) if has_content(g)][:10])

Top bigrams:
[('my skin', 4133), ('this product', 2534), ('my face', 1539), ('does not', 1137), ('do not', 1005), ('did not', 937), ('skin and', 887), ('love this', 859), ('it does', 771), ('use it', 722), ('using this', 600), ('to use', 584), ('using it', 524), ('sensitive skin', 511), ('been using', 467)]

Top trigrams:
[('it does not', 457), ('my skin is', 381), ('have been using', 376), ('on my face', 370), ('on my skin', 294), ('this product is', 284), ('love this product', 280), ('received this product', 271), ('and my skin', 246), ('in the morning', 220)]


In [44]:
V = len(uni)
def p_bigram(w1, w2):
    return (bi[(w1, w2)] + 1) / (uni[w1] + V)          # add-one smoothing

common_pair = bi.most_common(1)[0][0]
print("most common pair", common_pair, "-> P =", round(p_bigram(*common_pair), 5))
print("never-seen pair ('skin', 'spaceship') -> P =", round(p_bigram("skin", "spaceship"), 8), "(small but not zero)")

most common pair ('my', 'skin') -> P = 0.2043
never-seen pair ('skin', 'spaceship') -> P = 6.078e-05 (small but not zero)


## Phase 7: Bag of words

**Plain English:** a **bag of words** turns a sentence into counts of its words and ignores the order. "good scent" and "scent good" give the same bag. It is simple but powerful. Here we use it to see **which words are most common in the reviews**. The common content words show us what customers talk about, and that is how we choose the keywords for the aspect lexicon in Phase 8.

In [45]:
docs = sents["lemmas"].sample(min(40000, len(sents)), random_state=SEED).apply(" ".join)
bow = CountVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False, max_features=2000)
X = bow.fit_transform(docs)

totals = np.asarray(X.sum(axis=0)).ravel()
vocab = bow.get_feature_names_out()
top = pd.DataFrame({"word": vocab, "count": totals}).sort_values("count", ascending=False)
print("matrix shape (sentences x vocabulary):", X.shape)
print(top.head(30).T.to_string(header=False))

# one sentence as a bag
i = 0
print("\nSentence:", docs.iloc[i])
print("Bag     :", {vocab[j]: int(X[i, j]) for j in X[i].nonzero()[1]})

matrix shape (sentences x vocabulary): (40000, 2000)
word    skin   not   use  product    do  love  face  like  feel  good   dry  really   try  very  makeup  great  make  look  just  smell  moisturizer   day  work  recommend  leave  oily   get  little    go  cleanser
count  10642  8832  7373     6274  4459  3956  3311  3143  2905  2751  2466    2329  2138  2082    1919   1872  1851  1835  1796   1725         1675  1655  1652       1300   1293  1290  1267    1243  1235      1228

Sentence: skin felt firm week use
Bag     : {'skin': 1, 'felt': 1, 'firm': 1, 'week': 1, 'use': 1}


## Phase 8: Lexical semantics: the aspect lexicon

**Plain English:** an **aspect** is a topic people have opinions about (hydration, scent, price...). The **lexicon** lists, for each aspect, the words that signal it, including **synonyms** (*cheap, affordable, worth, value* all mean price/value). Because we matched on **lemmas**, one entry like `moisturize` covers *moisturizing*, *moisturizes* and *moisturized*.

**Word sense ambiguity (a classic NLP problem):** the word *dry* can be a skin type ("my dry skin") or a complaint ("it left my skin dry"). A keyword lexicon cannot tell the two senses apart, so we deliberately do **not** put *dry* in the lexicon. Real systems use word-sense disambiguation.

In [46]:
ASPECT_WORDS = {
    "hydration":     ["hydrate", "hydration", "moisturize", "moisture", "plump", "dewy", "nourish"],
    "texture":       ["texture", "creamy", "silky", "smooth", "greasy", "sticky", "thick", "lightweight", "heavy", "gel"],
    "scent":         ["smell", "scent", "fragrance", "fragrant", "aroma", "perfume", "odor"],
    "longevity":     ["last", "lasting", "longevity", "fade", "wear", "smudge", "linger", "sillage"],
    "value":         ["price", "expensive", "cheap", "affordable", "worth", "value", "money", "pricey", "cost", "overpriced"],
    "packaging":     ["packaging", "bottle", "pump", "tube", "jar", "container", "cap", "leak", "lid", "dispenser"],
    "irritation":    ["irritate", "irritation", "breakout", "burn", "sting", "redness", "rash", "allergic", "itchy", "clog", "react"],
    "effectiveness": ["effective", "result", "improve", "difference", "work", "glow", "brighten", "clear", "visible"],
    "absorption":    ["absorb", "absorption", "sink", "residue"],
    "coverage":      ["coverage", "pigment", "shade", "match", "blend", "buildable", "opaque", "finish"],
}
# lemmatise the lexicon itself so it always matches lemmatised text
LEXICON = {a: {lemma(w) for w in words} for a, words in ASPECT_WORDS.items()}

def find_aspects(lemmas):
    s = set(lemmas)
    return [a for a, words in LEXICON.items() if s & words]

sents["aspects"] = sents["lemmas"].apply(find_aspects)
tagged = sents["aspects"].str.len() > 0
print(f"{tagged.mean():.0%} of sentences mention at least one aspect")
print(sents["aspects"].explode().value_counts().to_string())
sents.loc[tagged, ["sentence", "aspects"]].head(8)

40% of sentences mention at least one aspect
aspects
effectiveness    4466
texture          3630
hydration        2878
scent            2779
value            2064
irritation       1957
packaging        1271
longevity        1032
absorption        730
coverage          392


,sentence,aspects
4,Tatcha is one of the very few brands with fragrance that my skin c...,[scent]
6,"Even now that my skin is dryer it still manages to hydrate, and st...",[hydration]
7,Only wish it were more affordable.,[value]
9,"Perfect for lips, cuticles and dry elbows, but it works anywhere r...",[effectiveness]
11,"Leaves skin smooth, no irritation.","[texture, irritation]"
15,"It turns to a flesh colored moisturizer after you rub it in, so yo...",[longevity]
19,I usually have dry skin and have to cake on moisturizer but this p...,[hydration]
22,"Overall, it does the job, but it is expensive for what it is.",[value]


## Phase 9: Sentiment per sentence

**Plain English:** for each sentence we want a score from **-1 (very negative)** to **+1 (very positive)**.

- **VADER** is a rule-based sentiment tool. It knows how positive each word is and handles negation (*not good*), intensifiers (*very good*), capitals and exclamation marks. We give it the **cleaned sentence with punctuation**, not the stop-word-removed version.
- We blend in the **star rating** of the review (25%). VADER can read a factual complaint like *"it barely lasts an hour"* as neutral, but the 1-star rating shows it is negative.
- Each sentence also gets a **weight** = 1 / (number of aspects it mentions). A sentence about three things is less clearly about any one of them.
- **Negated problems (a domain fix):** VADER reads the word *no* as negative, so it scores *"absorbs quickly with no residue"* as **bad**. But *no residue*, *no breakouts*, *did not irritate* are praise in skincare reviews. We add a small rule: if a negation word (*no, not, never, without*) is followed within 2 words by a problem word (*residue, breakout, irritate, sticky...*), the sentence counts as positive. *Dry* and *oily* are left out of the problem list on purpose, because they can also describe a skin type (*not good for oily skin* is not praise).

In [47]:
ALPHA = 0.25                     # weight of the star rating

vader = sents["sentence"].apply(lambda s: analyzer.polarity_scores(s)["compound"])

# negated problem rule: "no residue", "did not irritate", "no breakouts" = praise
NEGATORS = {"no", "not", "never", "without", "nothing", "zero"}
PROBLEMS = {lemma(w) for w in ["irritate", "irritation", "breakout", "redness", "rash", "residue", "sticky", "greasy",
                               "burn", "sting", "itchy", "itch", "clog", "flake", "pill", "cakey", "crack", "leak",
                               "fade", "smudge", "streak", "tight", "harsh", "chemical", "heavy"]}

def negated_problem(tokens):
    lem = [lemma(t) for t in tokens]
    return any(t in NEGATORS and any(x in PROBLEMS for x in lem[i + 1:i + 3]) for i, t in enumerate(tokens))

fires = sents["tokens"].apply(negated_problem)
sents["vader"] = np.where(fires, np.maximum(vader, 0.4), vader)     # raise to at least mildly positive
print(f"negated-problem rule fired on {fires.sum():,} sentences, for example:")
print(sents.loc[fires, "sentence"].head(5).to_string(index=False))
print()
stars_scaled = (sents["stars"] - 3) / 2          # 1..5 stars -> -1..+1
sents["score"] = (1 - ALPHA) * sents["vader"] + ALPHA * stars_scaled
sents["weight"] = sents["aspects"].apply(lambda a: 1 / max(len(a), 1))

print("Mean sentence score by star rating (should rise from 1 to 5):")
print(sents.groupby("stars")["score"].mean().round(3).to_string())
rho = spearmanr(sents["vader"], sents["stars"])[0]
print(f"\nSpearman correlation between the VADER score (with the negated-problem rule) and stars: {rho:.2f}")

negated-problem rule fired on 1,141 sentences, for example:
                                    Leaves skin smooth, no irritation.
    I mostly used this on my face and it did not feel sticky or heavy.
                                          And it wasn't greasy at all.
This product goes on smoothly and absorbs into my skin very quickly...
I have combo to dry skin and this stuff makes my skin feel amazing ...

Mean sentence score by star rating (should rise from 1 to 5):
stars
1   -0.244
2   -0.057
3    0.131
4    0.355
5    0.497

Spearman correlation between the VADER score (with the negated-problem rule) and stars: 0.19


## Phase 10: Ranking products for a request

**Plain English:**
1. Read the request, tokenise it, lemmatise it and find its **aspects** (the same Phase 8 lexicon).
2. For every product and aspect, average its sentence scores (weighted). We add a small safety number `K` so a product with one lucky sentence cannot beat one with many solid ones:

`aspect score = sum(score x weight) / (sum(weight) + K)`

3. A product's final score is the average over the requested aspects. We only rank products that have **at least `MIN_MENTIONS` sentences** about **every** requested aspect, so there is real evidence behind each quality.
4. An optional **category** filter (for example `"Moisturizers"`) limits the products considered.

In [48]:
K = 3                # smoothing strength
MIN_MENTIONS = 3     # evidence needed per requested aspect

def aspect_table(sentence_df):
    ex = sentence_df.explode("aspects").dropna(subset=["aspects"]).copy()
    ex["sw"] = ex["score"] * ex["weight"]
    g = ex.groupby(["product_id", "aspects"]).agg(sw=("sw", "sum"), w=("weight", "sum"), n=("score", "size"))
    g["aspect_score"] = g["sw"] / (g["w"] + K)
    return g.reset_index().rename(columns={"aspects": "aspect"})

ASP = aspect_table(sents)

def query_aspects(query):
    toks = remove_stop(tokenize(query))
    return find_aspects([lemma(t) for t in toks])

def rank_products(query, category=None, table=None):
    table = ASP if table is None else table
    wanted = query_aspects(query)
    if not wanted:
        return wanted, pd.DataFrame()
    sub = table[table["aspect"].isin(wanted)]
    pivot_s = sub.pivot(index="product_id", columns="aspect", values="aspect_score")
    pivot_n = sub.pivot(index="product_id", columns="aspect", values="n")
    for a in wanted:                                   # make sure every wanted column exists
        if a not in pivot_s: pivot_s[a] = np.nan; pivot_n[a] = 0
    pivot_n = pivot_n[wanted].fillna(0)
    ok = (pivot_n >= MIN_MENTIONS).all(axis=1)         # enough evidence on EVERY requested aspect
    out = pd.DataFrame({"score": pivot_s[wanted].mean(axis=1)})[ok].copy()
    for a in wanted:
        out[a] = pivot_s[a].round(2).astype(str) + " (n=" + pivot_n[a].astype(int).astype(str) + ")"
    out = out.reset_index().merge(products, on="product_id", how="left")
    if category:
        cat = category.lower()
        mask = (out["primary_category"].fillna("").str.lower().str.contains(cat, regex=False) |
                out["secondary_category"].fillna("").str.lower().str.contains(cat, regex=False) |
                out["tertiary_category"].fillna("").str.lower().str.contains(cat, regex=False))
        out = out[mask]
    return wanted, out.sort_values("score", ascending=False).reset_index(drop=True)

wanted, table = rank_products("hydrating and lightweight and good value")
print("Aspects found in the request:", wanted)
table[["product_name", "brand_name", "score"] + wanted].head(5)

Aspects found in the request: ['hydration', 'texture', 'value']


,product_name,brand_name,score,hydration,texture,value
0,Greek Yoghurt Foaming Cream Cleanser,KORRES,0.486543,0.47 (n=23),0.55 (n=40),0.44 (n=10)
1,Vitamin Enriched Face Base Priming Moisturizer,Bobbi Brown,0.460739,0.49 (n=90),0.49 (n=83),0.41 (n=24)
2,Green Clean Makeup Meltaway Cleansing Balm Limited Edition Jumbo,Farmacy,0.438717,0.42 (n=15),0.44 (n=29),0.46 (n=13)
3,GinZing Ultra-Hydrating Energy-Boosting Cream,Origins,0.428183,0.44 (n=70),0.45 (n=96),0.4 (n=11)
4,Rosebud Salve,Rosebud Perfume Co.,0.408971,0.4 (n=41),0.36 (n=36),0.47 (n=21)


## Phase 11: Evidence-backed explanation

**Plain English:** the system now has a winner. To **explain** the choice we pick real sentences from the winner's own reviews:

- **Evidence:** for each requested aspect, the most positive, most focused sentence (we prefer sentences of reasonable length, so the quote reads well).
- **Support level:** how many reviewers mentioned that aspect and what share were positive.
- **Opinion words:** the adjectives (Phase 5 POS tags) in the best sentences, as "reviewers describe it as ...".
- **Honest caveat:** the most negative sentence about the requested aspects, if there is a clearly negative one.

Nothing is generated or invented: every quote is copied from a review.

In [49]:
def aspect_sentences(pid, aspect):
    m = (sents["product_id"] == pid) & sents["aspects"].apply(lambda a: aspect in a)
    return sents[m]

def best_evidence(pid, aspect):
    sub = aspect_sentences(pid, aspect)
    sub = sub[(sub["sentence"].str.len() >= 25) & (sub["sentence"].str.len() <= 220)]   # readable quotes
    if sub.empty:
        return None
    return sub.assign(rank=sub["score"] * sub["weight"]).sort_values("rank", ascending=False).iloc[0]


def describe_words(pid, aspect, top_n=3):
    sub = aspect_sentences(pid, aspect).sort_values("score", ascending=False).head(25)
    counts = Counter(w for s in sub["sentence"] for w in opinion_adjectives(s))
    return [w for w, _ in counts.most_common(top_n)]

def recommend(query, category=None, runners=2, verbose=True):
    wanted, table = rank_products(query, category)
    if not wanted:
        if verbose: print("No known aspect in the request. Try: " + ", ".join(LEXICON))
        return None
    if table.empty:
        if verbose: print("No product has enough review evidence for all of:", ", ".join(wanted))
        return None
    top = table.iloc[0]
    pid = top["product_id"]
    lines = [f"Request     : {query}" + (f"   [category: {category}]" if category else ""),
             f"Aspects     : {', '.join(wanted)}",
             f"Recommended : {top['product_name']} ({top['brand_name']}), ${top['price_usd']:.0f}, avg rating {top['avg_rating']:.1f}",
             "", "Why:"]
    for a in wanted:
        ev = best_evidence(pid, a)
        sub = aspect_sentences(pid, a)
        share = (sub["score"] > 0.05).mean()
        if ev is None:
            lines.append(f"  - {a}: no suitable quote found")
            continue
        words = describe_words(pid, a)
        lines.append(f'  - {a}: "{ev["sentence"]}"  [review #{ev["review_id"]}, {ev["stars"]:.0f} stars]')
        lines.append(f"      {len(sub)} mentions, {share:.0%} positive" + (f"; described as: {', '.join(words)}" if words else ""))

    others = table.iloc[1:1 + runners]["product_name"].tolist()
    if others:
        lines += ["", "Runners-up  : " + " | ".join(others)]
    if verbose: print("\n".join(lines))
    return pid

cats = data.drop_duplicates("product_id")["tertiary_category"].value_counts()
print("Categories you can use:", list(cats.index[:6]))

Categories you can use: ['Moisturizers', 'Face Wash & Cleansers', 'Face Serums', 'Toners', 'Face Masks', 'Mists & Essences']


In [50]:
recommend("hydrating and lightweight and good value", category=cats.index[0])

Request     : hydrating and lightweight and good value   [category: Moisturizers]
Aspects     : hydration, texture, value
Recommended : Vitamin Enriched Face Base Priming Moisturizer (Bobbi Brown), $66, avg rating 4.6

Why:
  - hydration: "I'm so happy with this amazing face base - it moisturize and primes my face perfectly, leaves skin nourished and plump, the make up glides on like a dream."  [review #2822, 5 stars]
      90 mentions, 97% positive; described as: dry, good, super
  - texture: "This priming moisturizer feels so amazing on my skin , is lightweight and makes my foundation look so flawless!"  [review #4104, 5 stars]
      83 mentions, 96% positive; described as: great, smooth, oily
  - value: "This primer has definitely help keep my makeup on longer than it normally does so that definitely makes me feel happy & it's worth it!"  [review #5727, 5 stars]
      24 mentions, 92% positive; described as: worth, expensive, little

Runners-up  : GinZing Ultra-Hydrating Energy-Boos

'P270594'

In [51]:
# 1. Faithfulness
cleaned_reviews = dict(zip(data["review_id"], data["review_text"].apply(clean_text)))
checked = ok = 0
test_queries = ["hydrating and lightweight", "good scent", "worth the price", "does not irritate and works",
                "nice texture and absorbs quickly", "good packaging"]
for q in test_queries:
    wanted, table = rank_products(q)
    if table.empty: continue
    pid = table.iloc[0]["product_id"]
    for a in wanted:
        ev = best_evidence(pid, a)
        if ev is not None:
            checked += 1
            ok += ev["sentence"] in cleaned_reviews[ev["review_id"]]
print(f"Faithfulness: {ok}/{checked} quotes found word for word in the source review")

Faithfulness: 9/9 quotes found word for word in the source review


In [52]:
# 3. Stability: split-half reliability
rng = np.random.RandomState(SEED)
half = pd.Series(rng.rand(data["review_id"].max() + 1) < 0.5)             # random half per review
sents["half"] = sents["review_id"].map(half)
A = aspect_table(sents[sents["half"]])
B = aspect_table(sents[~sents["half"]])
m = A.merge(B, on=["product_id", "aspect"], suffixes=("_A", "_B"))
m = m[(m["n_A"] >= MIN_MENTIONS) & (m["n_B"] >= MIN_MENTIONS)]

rows = []
for a, g in m.groupby("aspect"):
    if len(g) >= 8:
        rows.append({"aspect": a, "products compared": len(g),
                     "rank correlation between halves": round(spearmanr(g["aspect_score_A"], g["aspect_score_B"])[0], 2)})
stab = pd.DataFrame(rows).sort_values("rank correlation between halves", ascending=False)
stab

,aspect,products compared,rank correlation between halves
7,scent,59,0.75
0,absorption,38,0.59
8,texture,67,0.59
3,hydration,57,0.38
2,effectiveness,68,0.37
5,longevity,52,0.36
9,value,65,0.34
4,irritation,62,0.28
1,coverage,13,0.24
6,packaging,54,0.14


In [53]:
# sanity: is the recommended product better rated than its category average?
rows = []
for q, c in [("hydrating and lightweight", cats.index[0]), ("good value", cats.index[0]),
             ("good scent", cats.index[min(1, len(cats) - 1)]), ("effective and does not irritate", cats.index[0])]:
    wanted, table = rank_products(q, c)
    if table.empty: continue
    pick = table.iloc[0]
    cat_avg = products[products["tertiary_category"].fillna("").str.lower().str.contains(c.lower(), regex=False)]["avg_rating"].mean()
    rows.append({"request": q, "category": c, "pick": pick["product_name"][:40],
                 "pick avg rating": round(pick["avg_rating"], 2), "category avg rating": round(cat_avg, 2)})
pd.DataFrame(rows)

,request,category,pick,pick avg rating,category avg rating
0,hydrating and lightweight,Moisturizers,Vitamin Enriched Face Base Priming Moist,4.61,4.26
1,good value,Moisturizers,Ultra Repair Cream Intense Hydration,4.52,4.26
2,good scent,Face Wash & Cleansers,Greek Yoghurt Foaming Cream Cleanser,4.57,4.29
3,effective and does not irritate,Moisturizers,Revitalizing Supreme+ Youth Power Creme,4.32,4.26


## Phase 13: Live demo

Type a request and press Enter. It understands these aspects: **hydration, texture, scent, longevity, value, packaging, irritation, effectiveness, absorption, coverage**. You can combine several in one sentence. Use the optional category to narrow the search.

In [58]:
query = input("What do you want from the product? ")
category = input("Category (press Enter to skip): ").strip() or None
recommend(query, category)

What do you want from the product? hydration
Category (press Enter to skip): face wash
Request     : hydration   [category: face wash]
Aspects     : hydration
Recommended : Greek Yoghurt Foaming Cream Cleanser (KORRES), $28, avg rating 4.6

Why:
  - hydration: "I also love combining this with my korres rose oil at night for a super clean and hydrated youthful skin look."  [review #2474, 5 stars]
      23 mentions, 96% positive; described as: clean, dry, amazing

Runners-up  : Take The Day Off Cleansing Balm Makeup Remover | Green Clean Makeup Meltaway Cleansing Balm Limited Edition Jumbo


'P456412'

In [55]:
# ready-made examples (change the text freely)
recommend("hydrating, lightweight and good value", category=cats.index[0])
print("\n" + "-" * 70 + "\n")
recommend("good scent and long lasting", category=cats.index[min(1, len(cats) - 1)])

Request     : hydrating, lightweight and good value   [category: Moisturizers]
Aspects     : hydration, texture, value
Recommended : Vitamin Enriched Face Base Priming Moisturizer (Bobbi Brown), $66, avg rating 4.6

Why:
  - hydration: "I'm so happy with this amazing face base - it moisturize and primes my face perfectly, leaves skin nourished and plump, the make up glides on like a dream."  [review #2822, 5 stars]
      90 mentions, 97% positive; described as: dry, good, super
  - texture: "This priming moisturizer feels so amazing on my skin , is lightweight and makes my foundation look so flawless!"  [review #4104, 5 stars]
      83 mentions, 96% positive; described as: great, smooth, oily
  - value: "This primer has definitely help keep my makeup on longer than it normally does so that definitely makes me feel happy & it's worth it!"  [review #5727, 5 stars]
      24 mentions, 92% positive; described as: worth, expensive, little

Runners-up  : GinZing Ultra-Hydrating Energy-Boostin

'P456412'